In [3]:
import pandas as pd
from collections import Counter
import os
import time

# Character class definition
class Character:
    def __init__(self, name, birth=None, death=None, gender=None, hair=None, height=None, race=None, realm=None, spouse=None):
        self.name = name
        self.birth = birth
        self.death = death
        self.gender = gender
        self.hair = hair
        self.height = height
        self.race = race
        self.realm = realm
        self.spouse = spouse

    def __repr__(self):
        return f"<Character {self.name}>"

    def info(self):
        return {
            "name": self.name,
            "birth": self.birth,
            "death": self.death,
            "gender": self.gender,
            "hair": self.hair,
            "height": self.height,
            "race": self.race,
            "realm": self.realm,
            "spouse": self.spouse.name if isinstance(self.spouse, Character) else self.spouse
        }

# Load characters from CSV
def load_characters(csv_file):
    df = pd.read_csv(csv_file)
    characters = {}

    for _, row in df.iterrows():
        name = row.get('name')
        if not name:
            continue
        char = Character(
            name=name,
            birth=row.get('birth'),
            death=row.get('death'),
            gender=row.get('gender'),
            hair=row.get('hair'),
            height=row.get('height'),
            race=row.get('race'),
            realm=row.get('realm'),
            spouse=row.get('spouse')
        )
        characters[name] = char

    # Link spouses as Character objects if possible
    for char in characters.values():
        spouse_name = char.spouse
        if spouse_name and spouse_name in characters:
            char.spouse = characters[spouse_name]

    return characters

# Setup output directory
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

# Load characters
characters = load_characters("lotr.csv")

# Now you can run your queries timed with `time.time()`

# Query 1 timing
start_time = time.time()
query_output1 = pd.DataFrame(
    [char.info() for char in characters.values()
     if isinstance(char.gender, str) and char.gender.lower() == "male"]
)
query_output1.to_csv(f"{output_dir}/query1_male_characters.csv", index=False)
print(f"Query 1 executed in {time.time() - start_time:.4f} seconds")

# Query 2 timing
start_time = time.time()
race_counts = Counter(char.race for char in characters.values() if isinstance(char.race, str))
query_output2 = pd.DataFrame(race_counts.items(), columns=['Race', 'Count'])
query_output2.to_csv(f"{output_dir}/query2_race_counts.csv", index=False)
print(f"Query 2 executed in {time.time() - start_time:.4f} seconds")

# Query 3 timing
start_time = time.time()
married_characters = [
    {"name": char.name, "spouse_name": char.spouse.name}
    for char in characters.values()
    if isinstance(char.spouse, Character)
]
query_output3 = pd.DataFrame(married_characters)
query_output3.to_csv(f"{output_dir}/query3_characters_with_spouses.csv", index=False)
print(f"Query 3 executed in {time.time() - start_time:.4f} seconds")

# Query 4 timing
start_time = time.time()
query_output4 = pd.DataFrame(
    [char.info() for char in characters.values()
     if isinstance(char.realm, str) and char.realm.lower() == "shire"]
)
query_output4.to_csv(f"{output_dir}/query4_characters_in_shire.csv", index=False)
print(f"Query 4 executed in {time.time() - start_time:.4f} seconds")

# Query 5 timing
start_time = time.time()
def normalize_gender(g):
    if not isinstance(g, str):
        return "unknown"
    g_low = g.strip().lower()
    if g_low in {"male", "males", "most likely males"}:
        return "male"
    return g_low

gender_counts = Counter(
    normalize_gender(char.gender) for char in characters.values()
    if char.gender and str(char.gender).strip() != ""
)
query_output5 = pd.DataFrame(gender_counts.items(), columns=["Gender", "Count"])
query_output5.to_csv(f"{output_dir}/query5_gender_counts_normalized.csv", index=False)
print(f"Query 5 executed in {time.time() - start_time:.4f} seconds")


Query 1 executed in 0.0044 seconds
Query 2 executed in 0.0010 seconds
Query 3 executed in 0.0008 seconds
Query 4 executed in 0.0012 seconds
Query 5 executed in 0.0009 seconds
